In [3]:
from pathlib import Path
import pandas as pd
import duckdb
import csv

# =========================================================
# PATHS
# =========================================================

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"
ANSWERS = RAW / "answers"
PROCESSED = PROJECT_ROOT / "data" / "processed"

INSIDERS_FILE = ANSWERS / "insiders.csv"

GT_USERS_FILE = (
    PROCESSED /
    "ground_truth_users_r62.parquet"
)

GT_EVENTS_FILE = (
    PROCESSED /
    "ground_truth_events_r62.parquet"
)

assert INSIDERS_FILE.exists()

for i in range(1, 6):
    assert (
        ANSWERS / f"r6.2-{i}.csv"
    ).exists()

print("Answers folder:", ANSWERS)
print("Users output  :", GT_USERS_FILE)
print("Events output :", GT_EVENTS_FILE)

print(
    "\n✅ SUCCESS — Ground-truth setup ready."
)

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
Answers folder: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\raw\r6.2\answers
Users output  : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\ground_truth_users_r62.parquet
Events output : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\ground_truth_events_r62.parquet

✅ SUCCESS — Ground-truth setup ready.


In [5]:
# =========================================================
# PARSE DETAILED OFFICIAL ANSWER FILES
# =========================================================

event_rows = []

for scenario in range(1, 6):

    scenario_file = (
        ANSWERS /
        f"r6.2-{scenario}.csv"
    )

    with open(
        scenario_file,
        "r",
        encoding="utf-8",
        errors="replace",
        newline=""
    ) as f:

        reader = csv.reader(f)

        for row in reader:

            if not row:
                continue

            source_type = row[0].strip().lower()

            # -----------------------------
            # Common fields
            # -----------------------------

            record = {
                "scenario":
                    scenario,

                "answer_file":
                    scenario_file.name,

                "source_type":
                    source_type,

                "event_id":
                    row[1] if len(row) > 1 else None,

                "timestamp_raw":
                    row[2] if len(row) > 2 else None,

                "user":
                    row[3] if len(row) > 3 else None,

                "pc":
                    row[4] if len(row) > 4 else None,

                "activity":
                    None,

                "filename":
                    None,

                "url":
                    None,

                "email_to":
                    None,

                "email_cc":
                    None,

                "email_bcc":
                    None,

                "email_from":
                    None,

                "email_size":
                    None,

                "attachments":
                    None,

                "to_removable_media":
                    None,

                "from_removable_media":
                    None,

                "content_name":
                    None,

                "scenario_related":
                    1
            }


            # -----------------------------
            # LOGON
            # -----------------------------

            if source_type == "logon":

                if len(row) >= 6:
                    record["activity"] = row[5]


            # -----------------------------
            # DEVICE
            # -----------------------------

            elif source_type == "device":

                if len(row) >= 7:
                    record["activity"] = row[6]


            # -----------------------------
            # FILE
            # -----------------------------

            elif source_type == "file":

                if len(row) >= 11:

                    record["filename"] = row[5]
                    record["activity"] = row[6]

                    record[
                        "to_removable_media"
                    ] = row[7]

                    record[
                        "from_removable_media"
                    ] = row[8]

                    record["content_name"] = row[9]


            # -----------------------------
            # EMAIL
            # -----------------------------

            elif source_type == "email":

                if len(row) >= 14:

                    record["email_to"] = row[5]
                    record["email_cc"] = row[6]
                    record["email_bcc"] = row[7]
                    record["email_from"] = row[8]

                    record["activity"] = row[9]
                    record["email_size"] = row[10]
                    record["attachments"] = row[11]

                    record["content_name"] = row[12]


            # -----------------------------
            # HTTP
            # -----------------------------

            elif source_type == "http":

                if len(row) >= 9:

                    record["url"] = row[5]
                    record["activity"] = row[6]
                    record["content_name"] = row[7]


            event_rows.append(record)


gt_events = pd.DataFrame(
    event_rows
)


gt_events["timestamp"] = pd.to_datetime(
    gt_events["timestamp_raw"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)


gt_events["email_size"] = pd.to_numeric(
    gt_events["email_size"],
    errors="coerce"
)


print("Rows:", len(gt_events))

print(
    "\nEvents by scenario:"
)

print(
    gt_events[
        "scenario"
    ].value_counts()
     .sort_index()
)

print(
    "\nEvents by source type:"
)

print(
    gt_events[
        "source_type"
    ].value_counts()
)

assert len(gt_events) > 0

print(
    "\n✅ SUCCESS — Official detailed answer events parsed."
)

Rows: 470

Events by scenario:
scenario
1     22
2    242
3     68
4    134
5      4
Name: count, dtype: int64

Events by source type:
source_type
http      174
email     135
device     78
file       55
logon      28
Name: count, dtype: int64

✅ SUCCESS — Official detailed answer events parsed.


In [7]:
# =========================================================
# SAVE EVENT-LEVEL GROUND TRUTH
# =========================================================

events_path = str(
    GT_EVENTS_FILE
).replace("\\", "/")


con = duckdb.connect()

con.register(
    "gt_events_df",
    gt_events
)

con.execute(f"""
COPY (

    SELECT

        scenario,
        answer_file,
        source_type,
        event_id,
        timestamp,
        user,
        pc,
        activity,

        filename,
        url,

        email_to,
        email_cc,
        email_bcc,
        email_from,
        email_size,
        attachments,

        to_removable_media,
        from_removable_media,

        content_name,

        scenario_related

    FROM gt_events_df

    ORDER BY
        scenario,
        timestamp

)

TO '{events_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert GT_EVENTS_FILE.exists()

print("Created:")
print(GT_EVENTS_FILE)

print(
    "\n✅ SUCCESS — Event-level ground truth saved."
)

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\ground_truth_events_r62.parquet

✅ SUCCESS — Event-level ground truth saved.


In [9]:
# =========================================================
# BUILD CLEAN USER-LEVEL GROUND TRUTH
# =========================================================

insiders = pd.read_csv(
    INSIDERS_FILE,
    dtype=str
)


r62_users = insiders[
    insiders["dataset"].astype(str)
    .str.strip()
    .eq("6.2")
].copy()


r62_users["start_original"] = (
    r62_users["start"]
)

r62_users["end_original"] = (
    r62_users["end"]
)


r62_users["start"] = pd.to_datetime(
    r62_users["start_original"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

r62_users["end"] = pd.to_datetime(
    r62_users["end_original"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)


# =========================================================
# FIRST / LAST OFFICIAL EVENT PER SCENARIO
# =========================================================

event_windows = (
    gt_events
    .groupby("scenario")["timestamp"]
    .agg(
        official_first_event="min",
        official_last_event="max"
    )
    .reset_index()
)


r62_users["scenario"] = (
    r62_users["scenario"]
    .astype(int)
)


r62_users = r62_users.merge(
    event_windows,
    on="scenario",
    how="left"
)


# =========================================================
# CORRECT ONLY INVALID START VALUES
# =========================================================

r62_users[
    "start_was_corrected"
] = r62_users["start"].isna().astype(int)


r62_users["start"] = (
    r62_users["start"]
    .fillna(
        r62_users[
            "official_first_event"
        ]
    )
)


r62_users[
    "is_original_malicious"
] = 1


gt_users = r62_users[
    [
        "dataset",
        "scenario",
        "details",
        "user",

        "start",
        "end",

        "start_original",
        "end_original",

        "start_was_corrected",

        "official_first_event",
        "official_last_event",

        "is_original_malicious"
    ]
].copy()


print(
    gt_users.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — User-level ground truth cleaned."
)

dataset  scenario    details    user               start                 end      start_original        end_original  start_was_corrected official_first_event official_last_event  is_original_malicious
    6.2         1 r6.2-1.csv ACM2278 2010-08-18 21:47:42 2010-08-24 03:48:51  8/18/2010 21:47:42 08/24/2010 03:48:51                    0  2010-08-18 21:47:42 2010-08-24 04:20:39                      1
    6.2         2 r6.2-2.csv CMP2946 2011-02-07 12:28:06 2011-03-04 12:30:25 02/07/2011 12:28:06 03/04/2011 12:30:25                    0  2011-02-02 09:08:00 2011-03-30 16:27:27                      1
    6.2         3 r6.2-3.csv PLJ1771 2010-08-12 08:09:00 2010-08-12 16:11:39  8/12/2010 08:09:00 08/12/2010 16:11:39                    0  2010-08-12 08:09:00 2010-08-18 10:49:48                      1
    6.2         4 r6.2-4.csv CDE1846 2011-02-21 11:43:39 2011-04-25 17:55:00   /21/2011 11:43:39 04/25/2011 17:55:00                    1  2011-02-21 11:43:39 2011-04-25 17:55:00              

In [11]:
# =========================================================
# SAVE USER-LEVEL GROUND TRUTH
# =========================================================

users_path = str(
    GT_USERS_FILE
).replace("\\", "/")


con = duckdb.connect()

con.register(
    "gt_users_df",
    gt_users
)

con.execute(f"""
COPY (

    SELECT *

    FROM gt_users_df

    ORDER BY scenario

)

TO '{users_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert GT_USERS_FILE.exists()

print("Created:")
print(GT_USERS_FILE)

print(
    "\n✅ SUCCESS — User-level ground truth saved."
)

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\ground_truth_users_r62.parquet

✅ SUCCESS — User-level ground truth saved.


In [13]:
# =========================================================
# GROUND TRUTH VALIDATION
# =========================================================

con = duckdb.connect()


user_summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(
        DISTINCT user
    ) AS malicious_users,

    COUNT(
        DISTINCT scenario
    ) AS scenarios,

    SUM(
        start_was_corrected
    ) AS corrected_starts

FROM read_parquet(
    '{users_path}'
)
""").df()


event_summary = con.execute(f"""
SELECT

    COUNT(*) AS events,

    COUNT(
        DISTINCT event_id
    ) AS unique_event_ids,

    COUNT(
        DISTINCT scenario
    ) AS scenarios,

    COUNT(
        DISTINCT user
    ) AS users_in_answer_events

FROM read_parquet(
    '{events_path}'
)
""").df()


null_event_keys = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{events_path}'
)

WHERE event_id IS NULL
   OR timestamp IS NULL
   OR user IS NULL
   OR source_type IS NULL
""").fetchone()[0]


con.close()


print("=" * 100)
print("GROUND TRUTH USERS")
print("=" * 100)

print(
    user_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 100)
print("GROUND TRUTH EVENTS")
print("=" * 100)

print(
    event_summary.to_string(
        index=False
    )
)

print(
    "\nNull critical event keys:",
    null_event_keys
)


assert int(
    user_summary.loc[0, "rows"]
) == 5

assert int(
    user_summary.loc[0, "malicious_users"]
) == 5

assert int(
    user_summary.loc[0, "scenarios"]
) == 5

assert int(
    user_summary.loc[0, "corrected_starts"]
) == 1

assert int(
    event_summary.loc[0, "events"]
) == int(
    event_summary.loc[0, "unique_event_ids"]
)

assert null_event_keys == 0


print(
    "\n✅ SUCCESS — OFFICIAL GROUND TRUTH PASSED ALL VALIDATION CHECKS."
)

GROUND TRUTH USERS
 rows  malicious_users  scenarios  corrected_starts
    5                5          5               1.0

GROUND TRUTH EVENTS
 events  unique_event_ids  scenarios  users_in_answer_events
    470               470          5                      29

Null critical event keys: 0

✅ SUCCESS — OFFICIAL GROUND TRUTH PASSED ALL VALIDATION CHECKS.


In [15]:
# =========================================================
# ADD PRIMARY-ACTOR VS CONTEXT LABELS
# =========================================================

# Map each scenario to its official malicious user
scenario_actor_map = (
    gt_users[
        ["scenario", "user"]
    ]
    .rename(
        columns={
            "user": "malicious_actor"
        }
    )
)

gt_events_labeled = (
    gt_events
    .merge(
        scenario_actor_map,
        on="scenario",
        how="left"
    )
)


# Event belongs to the primary malicious user
gt_events_labeled[
    "is_primary_actor_event"
] = (
    gt_events_labeled["user"]
    == gt_events_labeled["malicious_actor"]
).astype(int)


# Event is part of the scenario,
# but belongs to another contextual user
gt_events_labeled[
    "is_context_event"
] = (
    (
        gt_events_labeled["scenario_related"] == 1
    )
    &
    (
        gt_events_labeled[
            "is_primary_actor_event"
        ] == 0
    )
).astype(int)


print("=" * 100)
print("GROUND TRUTH EVENT ROLE SUMMARY")
print("=" * 100)

print(
    gt_events_labeled[
        [
            "scenario_related",
            "is_primary_actor_event",
            "is_context_event"
        ]
    ]
    .value_counts()
    .sort_index()
)

print("\nPrimary malicious users:")
print(
    gt_events_labeled[
        gt_events_labeled[
            "is_primary_actor_event"
        ] == 1
    ][
        ["scenario", "malicious_actor"]
    ]
    .drop_duplicates()
    .sort_values("scenario")
    .to_string(index=False)
)

print(
    "\n✅ SUCCESS — Primary-actor and context events separated."
)

GROUND TRUTH EVENT ROLE SUMMARY
scenario_related  is_primary_actor_event  is_context_event
1                 0                       1                    50
                  1                       0                   420
Name: count, dtype: int64

Primary malicious users:
 scenario malicious_actor
        1         ACM2278
        2         CMP2946
        3         PLJ1771
        4         CDE1846
        5         MBG3183

✅ SUCCESS — Primary-actor and context events separated.


In [17]:
# =========================================================
# SAVE REFINED EVENT-LEVEL GROUND TRUTH
# =========================================================

events_path = str(
    GT_EVENTS_FILE
).replace("\\", "/")


con = duckdb.connect()

con.register(
    "gt_events_labeled_df",
    gt_events_labeled
)

con.execute(f"""
COPY (

    SELECT

        scenario,
        answer_file,

        malicious_actor,

        source_type,
        event_id,
        timestamp,

        user,
        pc,
        activity,

        filename,
        url,

        email_to,
        email_cc,
        email_bcc,
        email_from,
        email_size,
        attachments,

        to_removable_media,
        from_removable_media,

        content_name,

        scenario_related,
        is_primary_actor_event,
        is_context_event

    FROM gt_events_labeled_df

    ORDER BY
        scenario,
        timestamp

)

TO '{events_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

print("Updated:")
print(GT_EVENTS_FILE)

print(
    "\n✅ SUCCESS — Refined event ground truth saved."
)

Updated:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\ground_truth_events_r62.parquet

✅ SUCCESS — Refined event ground truth saved.


In [19]:
# =========================================================
# REFINED GROUND TRUTH VALIDATION
# =========================================================

con = duckdb.connect()

role_summary = con.execute(f"""
SELECT

    COUNT(*) AS total_answer_events,

    SUM(
        is_primary_actor_event
    ) AS primary_actor_events,

    SUM(
        is_context_event
    ) AS context_events,

    COUNT(
        DISTINCT CASE
            WHEN is_primary_actor_event = 1
            THEN malicious_actor
        END
    ) AS malicious_actors

FROM read_parquet(
    '{events_path}'
)
""").df()


bad_role_rows = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{events_path}'
)

WHERE
    is_primary_actor_event
    + is_context_event
    <> 1
""").fetchone()[0]


wrong_actor = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{events_path}'
)

WHERE is_primary_actor_event = 1
  AND user <> malicious_actor
""").fetchone()[0]


con.close()


print("=" * 100)
print("REFINED GROUND TRUTH VALIDATION")
print("=" * 100)

print(
    role_summary.to_string(
        index=False
    )
)

print("\nRows with invalid role :", bad_role_rows)
print("Wrong primary actors   :", wrong_actor)


assert int(
    role_summary.loc[0, "total_answer_events"]
) == 470

assert int(
    role_summary.loc[0, "malicious_actors"]
) == 5

assert bad_role_rows == 0
assert wrong_actor == 0


print(
    "\n✅ SUCCESS — Refined ground truth roles are valid."
)

REFINED GROUND TRUTH VALIDATION
 total_answer_events  primary_actor_events  context_events  malicious_actors
                 470                 420.0            50.0                 5

Rows with invalid role : 0
Wrong primary actors   : 0

✅ SUCCESS — Refined ground truth roles are valid.
